# Google Colab VPS - Virtual DesktopTransform Google Colab into a full Linux desktop with remote access!## Features- Lightweight Linux Desktop (XFCE)- Remote Desktop Access (VNC)- 16 Hours CPU Daily- Persistent Storage (Google Drive)- Full VPS Control- No Disturbance - Your Private Workspace

## Step 1: Install Required Packages

In [ ]:
!apt-get update -qq!apt-get install -y -qq xfce4 xfce4-goodies!apt-get install -y -qq xrdp vnc4server!apt-get install -y -qq novnc!apt-get install -y -qq supervisor!apt-get install -y -qq nginx!apt-get install -y -qq wget curlprint("Desktop environment installed successfully!")

## Step 2: Install ngrok for Remote Access

In [ ]:
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz!tar -xzf ngrok-v3-stable-linux-amd64.tgz!mv ngrok /usr/local/bin/NGROK_TOKEN = "YOUR_NGROK_TOKEN"!ngrok config add-authtoken $NGROK_TOKENprint("ngrok installed successfully!")print("Please replace YOUR_NGROK_TOKEN with your actual token!")

## Step 3: Setup VNC Server

In [ ]:
import osVNC_PASSWORD = "colab123"!mkdir -p ~/.vnc!echo $VNC_PASSWORD | vncpasswd -f > ~/.vnc/passwd!chmod 600 ~/.vnc/passwdvnc_startup = """#!/bin/bashunset SESSION_MANAGERunset DBUS_SESSION_BUS_ADDRESS/usr/bin/startxfce4 &/usr/share/novnc/utils/launch.sh --vnc localhost:5901 --listen 0.0.0.0:6080 &"""with open("/root/.vnc/xstartup", "w") as f:    f.write(vnc_startup)!chmod +x ~/.vnc/xstartupprint("VNC server configured successfully!")print(f"VNC Password: {VNC_PASSWORD}")

## Step 4: Start VNC Server

In [ ]:
!vncserver -kill :1 2>/dev/null || true!vncserver :1 -geometry 1280x720 -depth 24print("VNC server started on display :1")print("Desktop is now running!")

## Step 5: Start ngrok Tunnel

In [ ]:
import subprocessimport time!pkill ngrok 2>/dev/null || truetime.sleep(2)ngrok_process = subprocess.Popen(["ngrok", "http", "6080"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)time.sleep(3)import requeststry:    response = requests.get("http://localhost:4040/api/tunnels")    tunnels = response.json()["tunnels"]    if tunnels:        public_url = tunnels[0]["public_url"]        print(f"VNC Desktop URL: {public_url}")        print(f"VNC Password: {VNC_PASSWORD}")        print("Click the link above to access your desktop!")    else:        print("No tunnels found.")except Exception as e:    print(f"Error: {e}")

## Step 6: Connect Google Drive

In [ ]:
from google.colab import drivedrive.mount("/content/drive")!mkdir -p /content/drive/MyDrive/colab_vpsprint("Google Drive mounted successfully!")

## Step 7: Install Additional Software

In [ ]:
!apt-get install -y -qq vim nano htop git curl wget unzip tree python3-pip python3-venv nodejs npmprint("Additional software installed!")

## Step 8: Setup Auto-Restart

In [ ]:
keep_alive_script = """#!/bin/bashwhile true; do    if ! pgrep -x "Xvnc" > /dev/null; then        echo "Restarting VNC server..."        vncserver :1 -geometry 1280x720 -depth 24    fi    if ! pgrep -x "ngrok" > /dev/null; then        echo "Restarting ngrok..."        ngrok http 6080 &    fi    sleep 60done"""with open("/root/keep_alive.sh", "w") as f:    f.write(keep_alive_script)!chmod +x /root/keep_alive.shsubprocess.Popen(["/root/keep_alive.sh"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)print("Auto-restart script started!")

## Step 9: Display Connection Info

In [ ]:
print("=" * 60)print("GOOGLE COLAB VPS - CONNECTION INFO")print("=" * 60)try:    response = requests.get("http://localhost:4040/api/tunnels")    tunnels = response.json()["tunnels"]    if tunnels:        print(f"VNC Desktop URL: {tunnels[0][public_url"]}")except:    print("Run the ngrok cell above to get URL")print(f"VNC Password: {VNC_PASSWORD}")print("Persistent Storage: /content/drive/MyDrive/colab_vps")print(f"CPU: {os.cpu_count()} cores")import psutilmem = psutil.virtual_memory()print(f"RAM: {mem.total / (1024**3):.1f} GB")print("=" * 60)